# 09_silver_gfs_forecast.ipynb — NOAA GFS / GFS-Wave Bronze → Silver

Este notebook procesa los forecast operacionales:

```text
data/NOAA GFS/gfs/gfs_YYYYMMDD_HHz/*.grib2
data/NOAA GFS-Wave/gfs_wave_YYYYMMDD_HHz/*.grib2
```

y genera:

```text
silver/forecast_gfs/source=NOAA_GFS/...
```

Tabla objetivo:

```text
run_time, target_time, horizon_hours,
zona_id, lat, lon,
wind_speed, wind_direction, pressure, temperature,
hs_forecast, tp_forecast, wave_direction_forecast, swell_forecast
```

Notas:
- Se muestrea cada forecast en las zonas/playas de `beach_geography`.
- `lat` y `lon` son las coordenadas de la zona/playa.
- Se conservan también las coordenadas de la celda GFS/GFS-Wave usada.
- No se imputan valores faltantes en Silver.
- GFS y GFS-Wave pueden tener rejillas diferentes; se muestrean por separado y luego se fusionan por `zona_id + run_time + horizon_hours`.

## Celda 0 — Montar Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Celda 1 — Instalar librerías necesarias

In [2]:
!pip -q install cfgrib eccodes xarray dask geopandas pyarrow shapely fiona tqdm pyproj

## Celda 2 — Imports, rutas y configuración

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import xarray as xr
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import cfgrib
import eccodes
import re
import unicodedata
import json
import shutil
import gc
from tqdm.auto import tqdm
from pyproj import Geod

BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
BRONZE_DIR = BASE_DIR / "data/bronze"
SILVER_DIR = BASE_DIR / "silver"

GFS_ROOT = BRONZE_DIR / "NOAA GFS" / "gfs"
GFS_WAVE_ROOT = BRONZE_DIR / "NOAA GFS-Wave"
DIM_ZONE_PATH = SILVER_DIR / "beach_geography" / "beach_geography.parquet"

OUT_DIR = SILVER_DIR / "forecast_gfs"
QC_DIR = SILVER_DIR / "_quality_reports"
META_DIR = SILVER_DIR / "_metadata"

for d in [OUT_DIR, QC_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SOURCE_NAME = "NOAA_GFS"

BBOX_CANARIAS = {
    "lat_min": 27.0,
    "lat_max": 29.5,
    "lon_min": -18.5,
    "lon_max": -13.0,
}

# Margen para leer/sampling de grillas.
BBOX_MARGIN_DEG = 0.75

MIN_VALID_TS = pd.Timestamp("2020-01-01", tz="UTC")
MAX_VALID_TS = pd.Timestamp("2035-01-01", tz="UTC")

# Para pruebas rápidas. Deja None para procesar todo.
MAX_FILES_FOR_TEST = None
MAX_ZONES_FOR_TEST = None

print("GFS_ROOT existe:", GFS_ROOT.exists())
print("GFS_WAVE_ROOT existe:", GFS_WAVE_ROOT.exists())
print("DIM_ZONE_PATH existe:", DIM_ZONE_PATH.exists())

if not DIM_ZONE_PATH.exists():
    raise FileNotFoundError("No existe beach_geography.parquet. Ejecuta primero 01_silver_dim_zone.ipynb.")

GFS_ROOT existe: True
GFS_WAVE_ROOT existe: True
DIM_ZONE_PATH existe: True


## Celda 3 — Utilidades generales

In [4]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(c for c in value if not unicodedata.combining(c))
    value = re.sub(r"\s+", " ", value)
    return value.upper()


def normalize_col(col):
    col = normalize_text(col)
    if pd.isna(col):
        return ""
    col = re.sub(r"[^A-Z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col


def standardize_longitudes_to_180(lon_values):
    lon = np.asarray(lon_values, dtype="float64")
    return ((lon + 180) % 360) - 180


def ensure_utc(x):
    return pd.to_datetime(x, utc=True, errors="coerce")


def wind_speed_from_uv(u, v):
    return np.sqrt(u ** 2 + v ** 2)


def wind_direction_from_uv(u, v):
    """
    Dirección meteorológica desde donde sopla el viento.
    u/v son componentes hacia este/norte.
    """
    return (np.degrees(np.arctan2(-u, -v)) + 360) % 360


def remove_existing_source_partition(base_dir, source_name=SOURCE_NAME):
    source_path = base_dir / f"source={source_name}"
    if source_path.exists():
        shutil.rmtree(source_path)
        print("Eliminada partición antigua:", source_path)


def write_partitioned_parquet(df, base_dir):
    if df.empty:
        print("DataFrame vacío. No se guarda.")
        return

    df = df.copy()
    df["source"] = df["source"].fillna(SOURCE_NAME).astype(str)
    df["run_date"] = pd.to_datetime(df["run_time"], utc=True).dt.strftime("%Y-%m-%d")
    df["year"] = pd.to_datetime(df["target_time"], utc=True).dt.year.astype("int64")

    table = pa.Table.from_pandas(df, preserve_index=False)

    pq.write_to_dataset(
        table,
        root_path=str(base_dir),
        partition_cols=["source", "run_date"],
        compression="snappy",
    )


def dataset_count_and_sample(path, source_name=SOURCE_NAME, sample_n=5):
    if not path.exists():
        return 0, pd.DataFrame()

    dataset = ds.dataset(str(path), format="parquet", partitioning="hive")
    count = dataset.count_rows(filter=(ds.field("source") == source_name))

    if count == 0:
        return 0, pd.DataFrame()

    sample = dataset.head(sample_n, filter=(ds.field("source") == source_name)).to_pandas()
    return count, sample


GEOD = Geod(ellps="WGS84")


def distance_m_between(lon1, lat1, lon2, lat2):
    _, _, dist = GEOD.inv(lon1, lat1, lon2, lat2)
    return dist

## Celda 4 — Cargar zonas costeras

In [5]:
beach_geography = pd.read_parquet(DIM_ZONE_PATH)

required_zone_cols = ["zona_id", "nombre_zona", "isla", "municipio", "lat", "lon"]
missing_zone_cols = [c for c in required_zone_cols if c not in beach_geography.columns]

if missing_zone_cols:
    raise ValueError(f"Faltan columnas en beach_geography: {missing_zone_cols}")

if MAX_ZONES_FOR_TEST is not None:
    beach_geography = beach_geography.head(MAX_ZONES_FOR_TEST).copy()

zones = beach_geography[required_zone_cols].dropna(subset=["zona_id", "lat", "lon"]).copy()
zones = zones.drop_duplicates(subset=["zona_id"]).reset_index(drop=True)

print("Zonas:", zones.shape)
display(zones.head())

Zonas: (561, 6)


,zona_id,nombre_zona,isla,municipio,lat,lon
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990


## Celda 5 — Localizar runs y archivos GFS/GFS-Wave

In [6]:
GFS_FILE_RE = re.compile(
    r"gfs_canarias_(?P<date>\d{8})_(?P<hour>\d{2})z_f(?P<horizon>\d{3})\.grib2$",
    re.IGNORECASE,
)

GFS_WAVE_FILE_RE = re.compile(
    r"gfs_wave_canarias_(?P<date>\d{8})_(?P<hour>\d{2})z_f(?P<horizon>\d{3})\.grib2$",
    re.IGNORECASE,
)


def parse_forecast_filename(path, kind):
    path = Path(path)
    regex = GFS_FILE_RE if kind == "gfs" else GFS_WAVE_FILE_RE
    m = regex.match(path.name)

    if not m:
        return {
            "kind": kind,
            "filename": path.name,
            "path": str(path),
            "run_time": pd.NaT,
            "horizon_hours": np.nan,
            "target_time": pd.NaT,
            "filename_parse_ok": False,
        }

    d = m.groupdict()
    run_time = pd.to_datetime(d["date"] + d["hour"], format="%Y%m%d%H", utc=True, errors="coerce")
    horizon = int(d["horizon"])
    target_time = run_time + pd.to_timedelta(horizon, unit="h")

    return {
        "kind": kind,
        "filename": path.name,
        "path": str(path),
        "run_time": run_time,
        "horizon_hours": horizon,
        "target_time": target_time,
        "filename_parse_ok": True,
    }


gfs_files = sorted(GFS_ROOT.rglob("*.grib2")) if GFS_ROOT.exists() else []
gfs_wave_files = sorted(GFS_WAVE_ROOT.rglob("*.grib2")) if GFS_WAVE_ROOT.exists() else []

if MAX_FILES_FOR_TEST is not None:
    gfs_files = gfs_files[:MAX_FILES_FOR_TEST]
    gfs_wave_files = gfs_wave_files[:MAX_FILES_FOR_TEST]

gfs_files_df = pd.DataFrame([parse_forecast_filename(p, "gfs") for p in gfs_files])
gfs_wave_files_df = pd.DataFrame([parse_forecast_filename(p, "gfs_wave") for p in gfs_wave_files])

print("Archivos GFS:", len(gfs_files_df))
display(gfs_files_df.head())

print("Archivos GFS-Wave:", len(gfs_wave_files_df))
display(gfs_wave_files_df.head())

if len(gfs_files_df) == 0 and len(gfs_wave_files_df) == 0:
    raise FileNotFoundError("No se encontraron archivos GFS ni GFS-Wave .grib2.")

all_forecast_files_df = pd.concat([gfs_files_df, gfs_wave_files_df], ignore_index=True)
all_forecast_files_df.to_csv(META_DIR / "gfs_forecast_files.csv", index=False)

print("Runs detectados:")
display(
    all_forecast_files_df
    .dropna(subset=["run_time"])
    .groupby(["kind", "run_time"])
    .agg(files=("filename", "count"), min_horizon=("horizon_hours", "min"), max_horizon=("horizon_hours", "max"))
    .reset_index()
)

Archivos GFS: 17


,kind,filename,path,run_time,horizon_hours,target_time,filename_parse_ok
0,gfs,gfs_canarias_20260508_12z_f000.grib2,/content/drive/MyDrive/AI Projects/DeepWave Ca...,2026-05-08 12:00:00+00:00,0,2026-05-08 12:00:00+00:00,True
1,gfs,gfs_canarias_20260508_12z_f003.grib2,/content/drive/MyDrive/AI Projects/DeepWave Ca...,2026-05-08 12:00:00+00:00,3,2026-05-08 15:00:00+00:00,True
2,gfs,gfs_canarias_20260508_12z_f006.grib2,/content/drive/MyDrive/AI Projects/DeepWave Ca...,2026-05-08 12:00:00+00:00,6,2026-05-08 18:00:00+00:00,True
3,gfs,gfs_canarias_20260508_12z_f009.grib2,/content/drive/MyDrive/AI Projects/DeepWave Ca...,2026-05-08 12:00:00+00:00,9,2026-05-08 21:00:00+00:00,True
4,gfs,gfs_canarias_20260508_12z_f012.grib2,/content/drive/MyDrive/AI Projects/DeepWave Ca...,2026-05-08 12:00:00+00:00,12,2026-05-09 00:00:00+00:00,True


Archivos GFS-Wave: 17


,kind,filename,path,run_time,horizon_hours,target_time,filename_parse_ok
0,gfs_wave,gfs_wave_canarias_20260508_12z_f000.grib2,/content/drive/MyDrive/AI Projects/DeepWave Ca...,2026-05-08 12:00:00+00:00,0,2026-05-08 12:00:00+00:00,True
1,gfs_wave,gfs_wave_canarias_20260508_12z_f003.grib2,/content/drive/MyDrive/AI Projects/DeepWave Ca...,2026-05-08 12:00:00+00:00,3,2026-05-08 15:00:00+00:00,True
2,gfs_wave,gfs_wave_canarias_20260508_12z_f006.grib2,/content/drive/MyDrive/AI Projects/DeepWave Ca...,2026-05-08 12:00:00+00:00,6,2026-05-08 18:00:00+00:00,True
3,gfs_wave,gfs_wave_canarias_20260508_12z_f009.grib2,/content/drive/MyDrive/AI Projects/DeepWave Ca...,2026-05-08 12:00:00+00:00,9,2026-05-08 21:00:00+00:00,True
4,gfs_wave,gfs_wave_canarias_20260508_12z_f012.grib2,/content/drive/MyDrive/AI Projects/DeepWave Ca...,2026-05-08 12:00:00+00:00,12,2026-05-09 00:00:00+00:00,True


Runs detectados:


,kind,run_time,files,min_horizon,max_horizon
0,gfs,2026-05-08 12:00:00+00:00,17,0,48
1,gfs_wave,2026-05-08 12:00:00+00:00,17,0,48


## Celda 6 — Alias de variables GRIB

In [7]:
GFS_VAR_ALIASES = {
    "u10": [
        "u10",
        "10u",
        "UGRD",
        "ugrd",
        "10 metre U wind component",
        "10m_u_component_of_wind",
        "u_component_of_wind_10m",
    ],
    "v10": [
        "v10",
        "10v",
        "VGRD",
        "vgrd",
        "10 metre V wind component",
        "10m_v_component_of_wind",
        "v_component_of_wind_10m",
    ],
    "temperature": [
        "t2m",
        "2t",
        "TMP",
        "tmp",
        "2 metre temperature",
        "2m_temperature",
        "temperature_2m",
    ],
    "pressure": [
        "sp",
        "pres",
        "PRES",
        "surface_pressure",
        "prmsl",
        "PRMSL",
        "msl",
        "MSL",
        "mean_sea_level_pressure",
    ],
    "precipitation_rate": [
        "prate",
        "PRATE",
        "precipitation_rate",
    ],
}

GFS_WAVE_VAR_ALIASES = {
    "hs_forecast": [
        "swh",
        "SWH",
        "htsgw",
        "HTSGW",
        "significant_height_of_combined_wind_waves_and_swell",
        "significant_wave_height",
    ],
    "tp_forecast": [
        "perpw",
        "PERPW",
        "pp1d",
        "PWPER",
        "peak_wave_period",
        "primary_wave_period",
    ],
    "wave_direction_forecast": [
        "dirpw",
        "DIRPW",
        "dirpws",
        "MWD",
        "mean_wave_direction",
        "primary_wave_direction",
    ],
    "swell_forecast": [
        "swell",
        "SWELL",
        "swheight",
        "swell_height",
        "swhsw",
    ],
    "swell_period_forecast": [
        "swper",
        "SWPER",
        "swell_period",
    ],
    "swell_direction_forecast": [
        "swdir",
        "SWDIR",
        "swell_direction",
    ],
}

## Celda 7 — Lectura robusta GRIB con cfgrib

In [8]:
CORE_COORDS = {
    "time",
    "valid_time",
    "step",
    "latitude",
    "longitude",
    "lat",
    "lon",
    "surface",
    "heightAboveGround",
}


def find_var_name_in_dataset(ds_in, aliases):
    available = list(ds_in.data_vars)

    for alias in aliases:
        if alias in available:
            return alias

    norm_map = {normalize_col(v): v for v in available}

    for alias in aliases:
        alias_norm = normalize_col(alias)
        if alias_norm in norm_map:
            return norm_map[alias_norm]

    # Buscar también en atributos cfVarName/GRIB_shortName/name.
    for var in available:
        attrs = ds_in[var].attrs
        attr_candidates = [
            attrs.get("GRIB_shortName"),
            attrs.get("GRIB_name"),
            attrs.get("long_name"),
            attrs.get("standard_name"),
        ]

        attr_norms = {normalize_col(x) for x in attr_candidates if x is not None}

        for alias in aliases:
            if normalize_col(alias) in attr_norms:
                return var

    return None


def clean_dataarray_for_merge(da, canonical_name):
    """
    Deja cada variable como DataArray 2D lat/lon.
    Si hay dimensiones extra de tamaño 1 o niveles, toma índice 0.
    """
    da = da.copy()

    coord_rename = {}
    if "latitude" in da.coords or "latitude" in da.dims:
        coord_rename["latitude"] = "lat"
    if "longitude" in da.coords or "longitude" in da.dims:
        coord_rename["longitude"] = "lon"

    if coord_rename:
        da = da.rename(coord_rename)

    # Eliminar dimensiones extra.
    for dim in list(da.dims):
        if dim not in ["lat", "lon"]:
            da = da.isel({dim: 0}, drop=True)

    # Eliminar coords escalares problemáticas.
    drop_coords = []

    for c in list(da.coords):
        if c not in ["lat", "lon"]:
            drop_coords.append(c)

    if drop_coords:
        da = da.drop_vars(drop_coords, errors="ignore")

    da.name = canonical_name

    return da


def open_grib_dataset_canonical(path, alias_map, source_label):
    """
    Abre un GRIB2 con cfgrib.open_datasets y extrae variables canónicas.
    Devuelve un Dataset con coords lat/lon y variables renombradas.
    """
    path = Path(path)

    try:
        grib_datasets = cfgrib.open_datasets(
            str(path),
            backend_kwargs={
                "indexpath": "",
            },
        )
    except Exception as e:
        raise ValueError(f"{source_label} {path.name}: cfgrib no pudo abrir el archivo: {repr(e)}")

    if not grib_datasets:
        raise ValueError(f"{source_label} {path.name}: cfgrib devolvió 0 datasets.")

    selected_arrays = {}
    dataset_var_inventory = []

    for ds_part in grib_datasets:
        dataset_var_inventory.append(list(ds_part.data_vars))

        for canonical, aliases in alias_map.items():
            if canonical in selected_arrays:
                continue

            found = find_var_name_in_dataset(ds_part, aliases)

            if found is None:
                continue

            da = clean_dataarray_for_merge(ds_part[found], canonical)
            selected_arrays[canonical] = da

    if not selected_arrays:
        inventory = json.dumps(dataset_var_inventory, ensure_ascii=False)
        raise ValueError(f"{source_label} {path.name}: no se detectaron variables esperadas. Inventario={inventory}")

    ds_out = xr.merge(list(selected_arrays.values()), compat="override", join="outer")

    if "lat" not in ds_out.coords or "lon" not in ds_out.coords:
        raise ValueError(f"{source_label} {path.name}: no hay coords lat/lon tras limpiar variables.")

    # Longitudes a -180..180.
    lon_std = standardize_longitudes_to_180(ds_out["lon"].values)
    ds_out = ds_out.assign_coords(lon=lon_std)
    ds_out = ds_out.sortby("lon")
    ds_out = ds_out.sortby("lat")

    # Recorte con margen.
    ds_out = ds_out.sel(
        lat=slice(BBOX_CANARIAS["lat_min"] - BBOX_MARGIN_DEG, BBOX_CANARIAS["lat_max"] + BBOX_MARGIN_DEG),
        lon=slice(BBOX_CANARIAS["lon_min"] - BBOX_MARGIN_DEG, BBOX_CANARIAS["lon_max"] + BBOX_MARGIN_DEG),
    )

    if ds_out.sizes.get("lat", 0) == 0 or ds_out.sizes.get("lon", 0) == 0:
        raise ValueError(f"{source_label} {path.name}: recorte bbox vacío.")

    ds_out = ds_out.load()

    meta = {
        "available_canonical_vars": list(ds_out.data_vars),
        "raw_dataset_inventory": dataset_var_inventory,
        "grid_lat_size": int(ds_out.sizes.get("lat", 0)),
        "grid_lon_size": int(ds_out.sizes.get("lon", 0)),
    }

    return ds_out, meta

## Celda 8 — Muestreo de grilla a zonas

In [9]:
def sample_dataset_to_zones(ds_in, zones_df, variables, grid_prefix):
    """
    Muestrea por nearest-neighbor las variables disponibles en las coordenadas de cada zona.
    """
    available = [v for v in variables if v in ds_in.data_vars]

    if not available:
        return pd.DataFrame()

    lat_points = xr.DataArray(zones_df["lat"].astype(float).values, dims="zone")
    lon_points = xr.DataArray(zones_df["lon"].astype(float).values, dims="zone")

    ds_sel = ds_in[available].sel(lat=lat_points, lon=lon_points, method="nearest")

    df = ds_sel.to_dataframe().reset_index()

    # La columna zone es índice posicional.
    df["zone"] = df["zone"].astype(int)

    # lat/lon que salen del dataset seleccionado son coords de la celda del modelo.
    if "lat" in df.columns:
        df = df.rename(columns={"lat": f"{grid_prefix}_grid_lat"})
    if "lon" in df.columns:
        df = df.rename(columns={"lon": f"{grid_prefix}_grid_lon"})

    zone_info = zones_df.reset_index(drop=True).reset_index().rename(columns={"index": "zone"})
    zone_info = zone_info.rename(columns={"lat": "lat", "lon": "lon"})

    df = df.merge(
        zone_info[
            [
                "zone",
                "zona_id",
                "nombre_zona",
                "isla",
                "municipio",
                "lat",
                "lon",
            ]
        ],
        on="zone",
        how="left",
    )

    if f"{grid_prefix}_grid_lat" in df.columns and f"{grid_prefix}_grid_lon" in df.columns:
        df[f"{grid_prefix}_grid_distance_km"] = [
            distance_m_between(
                lon1=row["lon"],
                lat1=row["lat"],
                lon2=row[f"{grid_prefix}_grid_lon"],
                lat2=row[f"{grid_prefix}_grid_lat"],
            ) / 1000.0
            for _, row in df.iterrows()
        ]
    else:
        df[f"{grid_prefix}_grid_distance_km"] = np.nan

    df = df.drop(columns=["zone"], errors="ignore")

    return df

## Celda 9 — Procesar archivos GFS atmosféricos

In [10]:
gfs_rows = []
gfs_file_summaries = []
gfs_errors = []

for _, row in tqdm(gfs_files_df.iterrows(), total=len(gfs_files_df), desc="Procesando GFS"):
    path = Path(row["path"])

    try:
        ds_gfs, meta = open_grib_dataset_canonical(
            path,
            alias_map=GFS_VAR_ALIASES,
            source_label="GFS",
        )

        df = sample_dataset_to_zones(
            ds_gfs,
            zones,
            variables=list(GFS_VAR_ALIASES.keys()),
            grid_prefix="gfs",
        )

        if df.empty:
            raise ValueError(f"{path.name}: sample_dataset_to_zones devolvió vacío.")

        df["run_time"] = row["run_time"]
        df["target_time"] = row["target_time"]
        df["horizon_hours"] = int(row["horizon_hours"])
        df["gfs_file"] = path.name

        # Derivados y unidades.
        if {"u10", "v10"}.issubset(df.columns):
            df["wind_speed"] = wind_speed_from_uv(df["u10"], df["v10"])
            df["wind_direction"] = wind_direction_from_uv(df["u10"], df["v10"])
        else:
            df["wind_speed"] = np.nan
            df["wind_direction"] = np.nan

        if "temperature" in df.columns:
            temp = pd.to_numeric(df["temperature"], errors="coerce")
            if temp.median(skipna=True) > 100:
                temp = temp - 273.15
            df["temperature"] = temp
        else:
            df["temperature"] = np.nan

        if "pressure" in df.columns:
            pressure = pd.to_numeric(df["pressure"], errors="coerce")
            if pressure.median(skipna=True) > 2000:
                pressure = pressure / 100.0
            df["pressure"] = pressure
        else:
            df["pressure"] = np.nan

        if "precipitation_rate" in df.columns:
            prate = pd.to_numeric(df["precipitation_rate"], errors="coerce")
            # kg/m2/s aproximadamente mm/s; pasamos a mm/h.
            if prate.quantile(0.99) < 1:
                df["precipitation_rate_mm_h"] = prate * 3600.0
            else:
                df["precipitation_rate_mm_h"] = prate
        else:
            df["precipitation_rate_mm_h"] = np.nan

        gfs_rows.append(df)

        gfs_file_summaries.append(
            {
                "filename": path.name,
                "run_time": row["run_time"],
                "target_time": row["target_time"],
                "horizon_hours": row["horizon_hours"],
                "rows": len(df),
                "available_vars": json.dumps(meta["available_canonical_vars"], ensure_ascii=False),
                "grid_lat_size": meta["grid_lat_size"],
                "grid_lon_size": meta["grid_lon_size"],
                "wind_speed_missing_pct": float(df["wind_speed"].isna().mean() * 100),
                "temperature_missing_pct": float(df["temperature"].isna().mean() * 100),
                "pressure_missing_pct": float(df["pressure"].isna().mean() * 100),
            }
        )

        ds_gfs.close()
        del df, ds_gfs
        gc.collect()

    except Exception as e:
        gfs_errors.append(
            {
                "filename": path.name,
                "path": str(path),
                "error": repr(e),
            }
        )

gfs_summary_df = pd.DataFrame(gfs_file_summaries)
gfs_errors_df = pd.DataFrame(gfs_errors)

print("Archivos GFS procesados:", len(gfs_summary_df))
print("Errores GFS:", len(gfs_errors_df))

display(gfs_summary_df.head())
display(gfs_errors_df)

gfs_summary_df.to_csv(QC_DIR / "quality_gfs_files_summary.csv", index=False)
gfs_errors_df.to_csv(QC_DIR / "quality_gfs_errors.csv", index=False)

if gfs_rows:
    gfs_forecast = pd.concat(gfs_rows, ignore_index=True)
else:
    gfs_forecast = pd.DataFrame()

print("gfs_forecast shape:", gfs_forecast.shape)

if len(gfs_errors_df):
    print("AVISO: hay errores en GFS. Si todos fallan, la validación final lo bloqueará.")

Procesando GFS:   0%|          | 0/17 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
/usr/local/lib/python3.12/dist-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
/usr/local/lib/python3.12/dist-packages/cfgrib

Archivos GFS procesados: 17
Errores GFS: 0


,filename,run_time,target_time,horizon_hours,rows,available_vars,grid_lat_size,grid_lon_size,wind_speed_missing_pct,temperature_missing_pct,pressure_missing_pct
0,gfs_canarias_20260508_12z_f000.grib2,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,561,"[""u10"", ""v10"", ""temperature"", ""pressure""]",14,27,0.0,0.0,0.0
1,gfs_canarias_20260508_12z_f003.grib2,2026-05-08 12:00:00+00:00,2026-05-08 15:00:00+00:00,3,561,"[""u10"", ""v10"", ""temperature"", ""pressure""]",14,27,0.0,0.0,0.0
2,gfs_canarias_20260508_12z_f006.grib2,2026-05-08 12:00:00+00:00,2026-05-08 18:00:00+00:00,6,561,"[""u10"", ""v10"", ""temperature"", ""pressure""]",14,27,0.0,0.0,0.0
3,gfs_canarias_20260508_12z_f009.grib2,2026-05-08 12:00:00+00:00,2026-05-08 21:00:00+00:00,9,561,"[""u10"", ""v10"", ""temperature"", ""pressure""]",14,27,0.0,0.0,0.0
4,gfs_canarias_20260508_12z_f012.grib2,2026-05-08 12:00:00+00:00,2026-05-09 00:00:00+00:00,12,561,"[""u10"", ""v10"", ""temperature"", ""pressure""]",14,27,0.0,0.0,0.0


""


gfs_forecast shape: (9537, 20)


## Celda 10 — Procesar archivos GFS-Wave

In [11]:
wave_rows = []
wave_file_summaries = []
wave_errors = []

for _, row in tqdm(gfs_wave_files_df.iterrows(), total=len(gfs_wave_files_df), desc="Procesando GFS-Wave"):
    path = Path(row["path"])

    try:
        ds_wave, meta = open_grib_dataset_canonical(
            path,
            alias_map=GFS_WAVE_VAR_ALIASES,
            source_label="GFS-Wave",
        )

        df = sample_dataset_to_zones(
            ds_wave,
            zones,
            variables=list(GFS_WAVE_VAR_ALIASES.keys()),
            grid_prefix="wave",
        )

        if df.empty:
            raise ValueError(f"{path.name}: sample_dataset_to_zones devolvió vacío.")

        df["run_time"] = row["run_time"]
        df["target_time"] = row["target_time"]
        df["horizon_hours"] = int(row["horizon_hours"])
        df["gfs_wave_file"] = path.name

        if "wave_direction_forecast" in df.columns:
            df["wave_direction_forecast"] = pd.to_numeric(df["wave_direction_forecast"], errors="coerce") % 360

        if "swell_direction_forecast" in df.columns:
            df["swell_direction_forecast"] = pd.to_numeric(df["swell_direction_forecast"], errors="coerce") % 360

        wave_rows.append(df)

        wave_file_summaries.append(
            {
                "filename": path.name,
                "run_time": row["run_time"],
                "target_time": row["target_time"],
                "horizon_hours": row["horizon_hours"],
                "rows": len(df),
                "available_vars": json.dumps(meta["available_canonical_vars"], ensure_ascii=False),
                "grid_lat_size": meta["grid_lat_size"],
                "grid_lon_size": meta["grid_lon_size"],
                "hs_forecast_missing_pct": float(df["hs_forecast"].isna().mean() * 100) if "hs_forecast" in df.columns else 100.0,
                "tp_forecast_missing_pct": float(df["tp_forecast"].isna().mean() * 100) if "tp_forecast" in df.columns else 100.0,
                "wave_direction_forecast_missing_pct": float(df["wave_direction_forecast"].isna().mean() * 100) if "wave_direction_forecast" in df.columns else 100.0,
            }
        )

        ds_wave.close()
        del df, ds_wave
        gc.collect()

    except Exception as e:
        wave_errors.append(
            {
                "filename": path.name,
                "path": str(path),
                "error": repr(e),
            }
        )

wave_summary_df = pd.DataFrame(wave_file_summaries)
wave_errors_df = pd.DataFrame(wave_errors)

print("Archivos GFS-Wave procesados:", len(wave_summary_df))
print("Errores GFS-Wave:", len(wave_errors_df))

display(wave_summary_df.head())
display(wave_errors_df)

wave_summary_df.to_csv(QC_DIR / "quality_gfs_wave_files_summary.csv", index=False)
wave_errors_df.to_csv(QC_DIR / "quality_gfs_wave_errors.csv", index=False)

if wave_rows:
    wave_forecast = pd.concat(wave_rows, ignore_index=True)
else:
    wave_forecast = pd.DataFrame()

print("wave_forecast shape:", wave_forecast.shape)

if len(wave_errors_df):
    print("AVISO: hay errores en GFS-Wave. Si todos fallan, la validación final lo bloqueará.")

Procesando GFS-Wave:   0%|          | 0/17 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
/usr/local/lib/python3.12/dist-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
/usr/local/lib/python3.12/dist-packages/cfgrib

Archivos GFS-Wave procesados: 17
Errores GFS-Wave: 0


,filename,run_time,target_time,horizon_hours,rows,available_vars,grid_lat_size,grid_lon_size,hs_forecast_missing_pct,tp_forecast_missing_pct,wave_direction_forecast_missing_pct
0,gfs_wave_canarias_20260508_12z_f000.grib2,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,561,"[""swell_direction_forecast"", ""hs_forecast"", ""t...",20,39,40.819964,40.819964,40.819964
1,gfs_wave_canarias_20260508_12z_f003.grib2,2026-05-08 12:00:00+00:00,2026-05-08 15:00:00+00:00,3,561,"[""swell_direction_forecast"", ""hs_forecast"", ""t...",20,39,40.819964,40.819964,40.819964
2,gfs_wave_canarias_20260508_12z_f006.grib2,2026-05-08 12:00:00+00:00,2026-05-08 18:00:00+00:00,6,561,"[""swell_direction_forecast"", ""hs_forecast"", ""t...",20,39,40.819964,40.819964,40.819964
3,gfs_wave_canarias_20260508_12z_f009.grib2,2026-05-08 12:00:00+00:00,2026-05-08 21:00:00+00:00,9,561,"[""swell_direction_forecast"", ""hs_forecast"", ""t...",20,39,40.819964,40.819964,40.819964
4,gfs_wave_canarias_20260508_12z_f012.grib2,2026-05-08 12:00:00+00:00,2026-05-09 00:00:00+00:00,12,561,"[""swell_direction_forecast"", ""hs_forecast"", ""t...",20,39,40.819964,40.819964,40.819964


""


wave_forecast shape: (9537, 17)


## Celda 11 — Fusionar GFS + GFS-Wave

In [12]:
merge_keys = ["zona_id", "run_time", "target_time", "horizon_hours"]

# Columnas atmosféricas que queremos conservar.
gfs_keep = [
    "zona_id",
    "nombre_zona",
    "isla",
    "municipio",
    "lat",
    "lon",
    "run_time",
    "target_time",
    "horizon_hours",
    "wind_speed",
    "wind_direction",
    "pressure",
    "temperature",
    "precipitation_rate_mm_h",
    "u10",
    "v10",
    "gfs_grid_lat",
    "gfs_grid_lon",
    "gfs_grid_distance_km",
    "gfs_file",
]

wave_keep = [
    "zona_id",
    "run_time",
    "target_time",
    "horizon_hours",
    "hs_forecast",
    "tp_forecast",
    "wave_direction_forecast",
    "swell_forecast",
    "swell_period_forecast",
    "swell_direction_forecast",
    "wave_grid_lat",
    "wave_grid_lon",
    "wave_grid_distance_km",
    "gfs_wave_file",
]

if not gfs_forecast.empty:
    for col in gfs_keep:
        if col not in gfs_forecast.columns:
            gfs_forecast[col] = np.nan

    gfs_part = gfs_forecast[gfs_keep].copy()
else:
    gfs_part = pd.DataFrame(columns=gfs_keep)

if not wave_forecast.empty:
    for col in wave_keep:
        if col not in wave_forecast.columns:
            wave_forecast[col] = np.nan

    wave_part = wave_forecast[wave_keep].copy()
else:
    wave_part = pd.DataFrame(columns=wave_keep)

if not gfs_part.empty and not wave_part.empty:
    forecast = gfs_part.merge(
        wave_part,
        on=merge_keys,
        how="outer",
        suffixes=("", "_wave"),
    )
elif not gfs_part.empty:
    forecast = gfs_part.copy()
    for col in wave_keep:
        if col not in forecast.columns:
            forecast[col] = np.nan
elif not wave_part.empty:
    forecast = wave_part.copy()

    # Añadir metadata de zona desde zones si solo hay wave.
    forecast = forecast.merge(
        zones[["zona_id", "nombre_zona", "isla", "municipio", "lat", "lon"]],
        on="zona_id",
        how="left",
    )

    for col in gfs_keep:
        if col not in forecast.columns:
            forecast[col] = np.nan
else:
    forecast = pd.DataFrame()

if forecast.empty:
    raise ValueError("No se pudo generar forecast: GFS y GFS-Wave están vacíos.")

forecast["source"] = SOURCE_NAME
forecast["temporal_resolution"] = "forecast"
forecast["forecast_type"] = "operational"
forecast["year"] = pd.to_datetime(forecast["target_time"], utc=True).dt.year.astype("Int64")

# Orden y duplicados.
forecast = (
    forecast
    .sort_values(["run_time", "horizon_hours", "zona_id"])
    .drop_duplicates(subset=["zona_id", "run_time", "target_time", "horizon_hours", "source"], keep="first")
    .reset_index(drop=True)
)

print("forecast shape:", forecast.shape)
print("Runs:", forecast["run_time"].nunique())
print("Horizons:", forecast["horizon_hours"].nunique())
print("Zonas:", forecast["zona_id"].nunique())

display(forecast.head())

forecast shape: (9537, 34)
Runs: 1
Horizons: 17
Zonas: 561


,zona_id,nombre_zona,isla,municipio,lat,lon,run_time,target_time,horizon_hours,wind_speed,...,swell_period_forecast,swell_direction_forecast,wave_grid_lat,wave_grid_lon,wave_grid_distance_km,gfs_wave_file,source,temporal_resolution,forecast_type,year
0,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,7.233931,...,NaN,357.839996,27.833335,-18.166627,8.604643,gfs_wave_canarias_20260508_12z_f000.grib2,NOAA_GFS,forecast,operational,2026
1,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,7.233931,...,NaN,357.839996,27.833335,-18.166627,10.830064,gfs_wave_canarias_20260508_12z_f000.grib2,NOAA_GFS,forecast,operational,2026
2,CAN_EH_CHARCO_AZUL_1,Charco Azul,El Hierro,Frontera,27.7616,-18.0404,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,7.233931,...,NaN,4.580000,27.833335,-17.999960,8.892389,gfs_wave_canarias_20260508_12z_f000.grib2,NOAA_GFS,forecast,operational,2026
3,CAN_EH_CHARCO_DE_LOS_SARGOS,Charco de los Sargos,El Hierro,Frontera,27.7846,-18.0119,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,7.233931,...,NaN,4.580000,27.833335,-17.999960,5.527301,gfs_wave_canarias_20260508_12z_f000.grib2,NOAA_GFS,forecast,operational,2026
4,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,7.157010,...,NaN,354.100006,27.666668,-18.166627,9.041978,gfs_wave_canarias_20260508_12z_f000.grib2,NOAA_GFS,forecast,operational,2026


## Celda 11B — Imputación sintética/proxy de nulos GFS/GFS-Wave

Esta celda reduce nulos usando los valores disponibles como referencia:

- Para `hs_forecast`, `tp_forecast` y `wave_direction_forecast`: relleno espacial por la zona válida más cercana dentro del mismo `run_time + horizon_hours`.
- Para restos sin vecino válido: mediana/media circular por isla/horizonte y horizonte.
- Para `swell_forecast`: si no existe en la descarga, se deriva de `hs_forecast`.
- Para `swell_period_forecast`: se deriva de `tp_forecast`.
- Para `swell_direction_forecast`: se deriva de `wave_direction_forecast`.
- Para `precipitation_rate_mm_h`: si no existe en la descarga, se genera un proxy conservador desde presión y viento.

Todo valor sintético queda marcado con:

```text
<variable>_was_synthetic = True
<variable>_imputation_method = ...
<variable>_flag = 3
```


In [13]:

SYNTHETIC_MAX_NEAREST_KM = 250.0

SYNTHETIC_VARS = [
    "hs_forecast",
    "tp_forecast",
    "wave_direction_forecast",
    "swell_forecast",
    "swell_period_forecast",
    "swell_direction_forecast",
    "precipitation_rate_mm_h",
]

original_missing_pct = {}

for col in SYNTHETIC_VARS:
    if col not in forecast.columns:
        forecast[col] = np.nan

    forecast[col] = pd.to_numeric(forecast[col], errors="coerce")
    original_missing_pct[col] = float(forecast[col].isna().mean() * 100)

    was_col = f"{col}_was_synthetic"
    method_col = f"{col}_imputation_method"

    forecast[was_col] = False
    forecast[method_col] = np.where(
        forecast[col].notna(),
        "original_model_value",
        "missing_original",
    )


def haversine_km(lon1, lat1, lon2, lat2):
    lon1 = np.deg2rad(float(lon1))
    lat1 = np.deg2rad(float(lat1))
    lon2 = np.deg2rad(np.asarray(lon2, dtype="float64"))
    lat2 = np.deg2rad(np.asarray(lat2, dtype="float64"))

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    return 6371.0 * c


def circular_mean_deg(values):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna()

    if len(values) == 0:
        return np.nan

    radians = np.deg2rad(values.values)
    sin_mean = np.sin(radians).mean()
    cos_mean = np.cos(radians).mean()

    if np.isclose(sin_mean, 0) and np.isclose(cos_mean, 0):
        return np.nan

    return float((np.degrees(np.arctan2(sin_mean, cos_mean)) + 360) % 360)


def mark_synthetic(df, idx, col, value, method):
    df.loc[idx, col] = value
    df.loc[idx, f"{col}_was_synthetic"] = True
    df.loc[idx, f"{col}_imputation_method"] = method


def nearest_fill_same_horizon(df, col, group_cols=("run_time", "horizon_hours"), max_km=SYNTHETIC_MAX_NEAREST_KM):
    """
    Rellena cada nulo con el valor de la zona válida más cercana en el mismo horizonte.
    """
    df = df.copy()

    if df[col].notna().sum() == 0:
        return df

    for _, g in df.groupby(list(group_cols), dropna=False, sort=False):
        valid = g[g[col].notna()].copy()
        missing = g[g[col].isna()].copy()

        if valid.empty or missing.empty:
            continue

        valid_lats = valid["lat"].astype(float).values
        valid_lons = valid["lon"].astype(float).values
        valid_values = valid[col].astype(float).values

        for idx, row in missing.iterrows():
            distances = haversine_km(
                lon1=row["lon"],
                lat1=row["lat"],
                lon2=valid_lons,
                lat2=valid_lats,
            )

            if len(distances) == 0 or np.all(~np.isfinite(distances)):
                continue

            nearest_pos = int(np.nanargmin(distances))
            nearest_distance = float(distances[nearest_pos])

            if np.isfinite(nearest_distance) and nearest_distance <= max_km:
                value = valid_values[nearest_pos]

                if pd.notna(value):
                    mark_synthetic(
                        df,
                        idx,
                        col,
                        value,
                        f"nearest_valid_same_horizon_{nearest_distance:.1f}km",
                    )

    return df


def fill_scalar_by_group_median(df, col, group_cols, method_name):
    df = df.copy()
    missing = df[col].isna()

    if not missing.any():
        return df

    group_median = df.groupby(group_cols, dropna=False)[col].transform("median")
    fill_mask = missing & group_median.notna()

    df.loc[fill_mask, col] = group_median[fill_mask]
    df.loc[fill_mask, f"{col}_was_synthetic"] = True
    df.loc[fill_mask, f"{col}_imputation_method"] = method_name

    return df


def fill_direction_by_group_circular_mean(df, col, group_cols, method_name):
    df = df.copy()
    missing = df[col].isna()

    if not missing.any():
        return df

    circular_values = {}

    for key, g in df.groupby(group_cols, dropna=False, sort=False):
        circular_values[key] = circular_mean_deg(g[col])

    if len(group_cols) == 1:
        keys = df[group_cols[0]]
    else:
        keys = list(zip(*[df[c] for c in group_cols]))

    fill_values = pd.Series(keys, index=df.index).map(circular_values)
    fill_mask = missing & fill_values.notna()

    df.loc[fill_mask, col] = fill_values[fill_mask]
    df.loc[fill_mask, f"{col}_was_synthetic"] = True
    df.loc[fill_mask, f"{col}_imputation_method"] = method_name

    return df


# 1) Oleaje principal: usar vecinos válidos del mismo horizonte.
for col in ["hs_forecast", "tp_forecast", "wave_direction_forecast"]:
    forecast = nearest_fill_same_horizon(
        forecast,
        col,
        group_cols=("run_time", "horizon_hours"),
        max_km=SYNTHETIC_MAX_NEAREST_KM,
    )

# 2) Fallback por isla/horizonte y horizonte.
for col in ["hs_forecast", "tp_forecast"]:
    forecast = fill_scalar_by_group_median(
        forecast,
        col,
        ["run_time", "horizon_hours", "isla"],
        "group_median_same_run_horizon_island",
    )

    forecast = fill_scalar_by_group_median(
        forecast,
        col,
        ["run_time", "horizon_hours"],
        "group_median_same_run_horizon",
    )

forecast = fill_direction_by_group_circular_mean(
    forecast,
    "wave_direction_forecast",
    ["run_time", "horizon_hours", "isla"],
    "circular_mean_same_run_horizon_island",
)

forecast = fill_direction_by_group_circular_mean(
    forecast,
    "wave_direction_forecast",
    ["run_time", "horizon_hours"],
    "circular_mean_same_run_horizon",
)

# 3) Swell: si existe algún valor real, usar vecinos/medianas; si no, derivar de hs.
if forecast["swell_forecast"].notna().sum() > 0:
    forecast = nearest_fill_same_horizon(
        forecast,
        "swell_forecast",
        group_cols=("run_time", "horizon_hours"),
        max_km=SYNTHETIC_MAX_NEAREST_KM,
    )

    forecast = fill_scalar_by_group_median(
        forecast,
        "swell_forecast",
        ["run_time", "horizon_hours", "isla"],
        "group_median_same_run_horizon_island",
    )

    forecast = fill_scalar_by_group_median(
        forecast,
        "swell_forecast",
        ["run_time", "horizon_hours"],
        "group_median_same_run_horizon",
    )

swell_missing = forecast["swell_forecast"].isna() & forecast["hs_forecast"].notna()

if swell_missing.any():
    wind = pd.to_numeric(forecast["wind_speed"], errors="coerce").fillna(forecast["wind_speed"].median())
    wave_dir = pd.to_numeric(forecast["wave_direction_forecast"], errors="coerce")

    # Más swell relativo con viento local menor y oleaje de componente N/NW/NE.
    ratio = 0.72 - 0.015 * wind
    atlantic_dir = (wave_dir >= 270) | (wave_dir <= 70)
    ratio = ratio + np.where(atlantic_dir.fillna(False), 0.05, 0.0)
    ratio = pd.Series(ratio, index=forecast.index).clip(lower=0.45, upper=0.82)

    forecast.loc[swell_missing, "swell_forecast"] = (
        forecast.loc[swell_missing, "hs_forecast"] * ratio.loc[swell_missing]
    )

    forecast.loc[swell_missing, "swell_forecast_was_synthetic"] = True
    forecast.loc[swell_missing, "swell_forecast_imputation_method"] = "derived_from_hs_wind_direction_proxy"

# 4) Swell period y direction desde tp/dirección si faltan.
swell_period_missing = forecast["swell_period_forecast"].isna() & forecast["tp_forecast"].notna()

if swell_period_missing.any():
    forecast.loc[swell_period_missing, "swell_period_forecast"] = (
        forecast.loc[swell_period_missing, "tp_forecast"] * 0.90
    ).clip(lower=3, upper=30)

    forecast.loc[swell_period_missing, "swell_period_forecast_was_synthetic"] = True
    forecast.loc[swell_period_missing, "swell_period_forecast_imputation_method"] = "derived_from_tp_proxy"

swell_dir_missing = forecast["swell_direction_forecast"].isna() & forecast["wave_direction_forecast"].notna()

if swell_dir_missing.any():
    forecast.loc[swell_dir_missing, "swell_direction_forecast"] = forecast.loc[
        swell_dir_missing,
        "wave_direction_forecast",
    ]

    forecast.loc[swell_dir_missing, "swell_direction_forecast_was_synthetic"] = True
    forecast.loc[swell_dir_missing, "swell_direction_forecast_imputation_method"] = "copied_from_wave_direction_proxy"

# 5) Precipitación: si no existe en GFS, generar proxy conservador con presión y viento.
if forecast["precipitation_rate_mm_h"].notna().sum() > 0:
    forecast = nearest_fill_same_horizon(
        forecast,
        "precipitation_rate_mm_h",
        group_cols=("run_time", "horizon_hours"),
        max_km=SYNTHETIC_MAX_NEAREST_KM,
    )

    forecast = fill_scalar_by_group_median(
        forecast,
        "precipitation_rate_mm_h",
        ["run_time", "horizon_hours", "isla"],
        "group_median_same_run_horizon_island",
    )

    forecast = fill_scalar_by_group_median(
        forecast,
        "precipitation_rate_mm_h",
        ["run_time", "horizon_hours"],
        "group_median_same_run_horizon",
    )

precip_missing = forecast["precipitation_rate_mm_h"].isna()

if precip_missing.any():
    pressure = pd.to_numeric(forecast["pressure"], errors="coerce")
    wind = pd.to_numeric(forecast["wind_speed"], errors="coerce")

    pressure_ref = pressure.median(skipna=True)

    if pd.isna(pressure_ref):
        pressure_ref = 1015.0

    # Proxy simple: precipitación aumenta con presión baja y viento moderado/fuerte.
    precip_proxy = (
        np.maximum(0, pressure_ref - pressure) * 0.18
        + np.maximum(0, wind - 7.0) * 0.04
    )

    precip_proxy = pd.Series(precip_proxy, index=forecast.index).fillna(0.0)
    precip_proxy = precip_proxy.mask(precip_proxy < 0.05, 0.0).clip(lower=0, upper=15)

    forecast.loc[precip_missing, "precipitation_rate_mm_h"] = precip_proxy.loc[precip_missing]
    forecast.loc[precip_missing, "precipitation_rate_mm_h_was_synthetic"] = True
    forecast.loc[precip_missing, "precipitation_rate_mm_h_imputation_method"] = "pressure_wind_proxy"

# Normalizar direcciones después de imputar.
for col in ["wave_direction_forecast", "swell_direction_forecast", "wind_direction"]:
    if col in forecast.columns:
        forecast[col] = pd.to_numeric(forecast[col], errors="coerce") % 360

synthetic_summary = []

for col in SYNTHETIC_VARS:
    synthetic_summary.append(
        {
            "variable": col,
            "original_missing_pct": original_missing_pct.get(col, np.nan),
            "final_missing_pct": float(forecast[col].isna().mean() * 100),
            "synthetic_pct": float(forecast[f"{col}_was_synthetic"].mean() * 100),
            "real_or_model_original_pct": float((~forecast[f"{col}_was_synthetic"] & forecast[col].notna()).mean() * 100),
            "methods": json.dumps(
                forecast[f"{col}_imputation_method"].value_counts(dropna=False).to_dict(),
                ensure_ascii=False,
            ),
        }
    )

synthetic_summary_df = pd.DataFrame(synthetic_summary)

display(synthetic_summary_df)

synthetic_summary_df.to_csv(
    QC_DIR / "quality_gfs_synthetic_imputation_summary.csv",
    index=False,
)

print("Imputación sintética/proxy GFS completada.")


,variable,original_missing_pct,final_missing_pct,synthetic_pct,real_or_model_original_pct,methods
0,hs_forecast,40.819964,0.0,40.819964,59.180036,"{""original_model_value"": 5644, ""nearest_valid_..."
1,tp_forecast,40.819964,0.0,40.819964,59.180036,"{""original_model_value"": 5644, ""nearest_valid_..."
2,wave_direction_forecast,40.819964,0.0,40.819964,59.180036,"{""original_model_value"": 5644, ""nearest_valid_..."
3,swell_forecast,100.000000,0.0,100.000000,0.000000,"{""derived_from_hs_wind_direction_proxy"": 9537}"
4,swell_period_forecast,100.000000,0.0,100.000000,0.000000,"{""derived_from_tp_proxy"": 9537}"
5,swell_direction_forecast,41.270840,0.0,41.270840,58.729160,"{""original_model_value"": 5601, ""copied_from_wa..."
6,precipitation_rate_mm_h,100.000000,0.0,100.000000,0.000000,"{""pressure_wind_proxy"": 9537}"


Imputación sintética/proxy GFS completada.


## Celda 12 — Flags de calidad

In [14]:

VARIABLE_RANGES = {
    "wind_speed": (0, 60),
    "wind_direction": (0, 360),
    "pressure": (800, 1100),
    "temperature": (-20, 55),
    "precipitation_rate_mm_h": (0, 300),
    "u10": (-60, 60),
    "v10": (-60, 60),
    "hs_forecast": (0, 20),
    "tp_forecast": (0, 40),
    "wave_direction_forecast": (0, 360),
    "swell_forecast": (0, 20),
    "swell_period_forecast": (0, 40),
    "swell_direction_forecast": (0, 360),
}


def add_quality_flags(df, variable_ranges):
    """
    Flags:
    0 = ok original/model
    1 = missing
    2 = outlier
    3 = synthetic/proxy/imputed
    """
    df = df.copy()

    for col, (vmin, vmax) in variable_ranges.items():
        if col not in df.columns:
            df[col] = np.nan

        flag_col = f"{col}_flag"
        synthetic_col = f"{col}_was_synthetic"

        df[flag_col] = 0

        if synthetic_col in df.columns:
            synthetic_mask = df[synthetic_col].fillna(False).astype(bool) & df[col].notna()
            df.loc[synthetic_mask, flag_col] = 3

        missing_mask = df[col].isna()
        outlier_mask = (~missing_mask) & ((df[col] < vmin) | (df[col] > vmax))

        # Missing/outlier tienen prioridad sobre synthetic.
        df.loc[missing_mask, flag_col] = 1
        df.loc[outlier_mask, flag_col] = 2
        df[flag_col] = df[flag_col].astype("int8")

    return df


forecast = add_quality_flags(forecast, VARIABLE_RANGES)

display(forecast.head())


,zona_id,nombre_zona,isla,municipio,lat,lon,run_time,target_time,horizon_hours,wind_speed,...,temperature_flag,precipitation_rate_mm_h_flag,u10_flag,v10_flag,hs_forecast_flag,tp_forecast_flag,wave_direction_forecast_flag,swell_forecast_flag,swell_period_forecast_flag,swell_direction_forecast_flag
0,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,7.233931,...,0,3,0,0,0,0,0,3,3,0
1,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,7.233931,...,0,3,0,0,0,0,0,3,3,0
2,CAN_EH_CHARCO_AZUL_1,Charco Azul,El Hierro,Frontera,27.7616,-18.0404,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,7.233931,...,0,3,0,0,0,0,0,3,3,0
3,CAN_EH_CHARCO_DE_LOS_SARGOS,Charco de los Sargos,El Hierro,Frontera,27.7846,-18.0119,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,7.233931,...,0,3,0,0,0,0,0,3,3,0
4,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,7.157010,...,0,3,0,0,0,0,0,3,3,0


## Celda 13 — Reportes de calidad

In [15]:
quality_summary = pd.DataFrame(
    [
        {
            "table": "forecast_gfs",
            "source": SOURCE_NAME,
            "rows": len(forecast),
            "runs": forecast["run_time"].nunique(),
            "horizons": forecast["horizon_hours"].nunique(),
            "zones": forecast["zona_id"].nunique(),
            "target_time_min": forecast["target_time"].min(),
            "target_time_max": forecast["target_time"].max(),
            "wind_speed_missing_pct": float(forecast["wind_speed"].isna().mean() * 100),
            "temperature_missing_pct": float(forecast["temperature"].isna().mean() * 100),
            "pressure_missing_pct": float(forecast["pressure"].isna().mean() * 100),
            "hs_forecast_missing_pct": float(forecast["hs_forecast"].isna().mean() * 100),
            "tp_forecast_missing_pct": float(forecast["tp_forecast"].isna().mean() * 100),
            "wave_direction_forecast_missing_pct": float(forecast["wave_direction_forecast"].isna().mean() * 100),
            "gfs_files_processed": len(gfs_summary_df),
            "gfs_wave_files_processed": len(wave_summary_df),
            "gfs_errors": len(gfs_errors_df),
            "gfs_wave_errors": len(wave_errors_df),
        }
    ]
)

missing_by_column = (
    forecast.isna()
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_pct"})
)

forecast_stats_cols = [
    "wind_speed",
    "wind_direction",
    "pressure",
    "temperature",
    "precipitation_rate_mm_h",
    "hs_forecast",
    "tp_forecast",
    "wave_direction_forecast",
    "swell_forecast",
]

stats_cols = [c for c in forecast_stats_cols if c in forecast.columns]

forecast_stats = (
    forecast[stats_cols]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
)

display(quality_summary)
display(missing_by_column)
display(forecast_stats)

quality_summary.to_csv(QC_DIR / "quality_gfs_forecast_summary.csv", index=False)
missing_by_column.to_csv(QC_DIR / "quality_gfs_forecast_missing_by_column.csv", index=False)
forecast_stats.to_csv(QC_DIR / "quality_gfs_forecast_stats.csv", index=False)

,table,source,rows,runs,horizons,zones,target_time_min,target_time_max,wind_speed_missing_pct,temperature_missing_pct,pressure_missing_pct,hs_forecast_missing_pct,tp_forecast_missing_pct,wave_direction_forecast_missing_pct,gfs_files_processed,gfs_wave_files_processed,gfs_errors,gfs_wave_errors
0,forecast_gfs,NOAA_GFS,9537,1,17,561,2026-05-08 12:00:00+00:00,2026-05-10 12:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,17,17,0,0


,column,missing_pct
0,zona_id,0.0
1,nombre_zona,0.0
2,isla,0.0
3,municipio,0.0
4,lat,0.0
...,...,...
56,tp_forecast_flag,0.0
57,wave_direction_forecast_flag,0.0
58,swell_forecast_flag,0.0
59,swell_period_forecast_flag,0.0


,variable,count,mean,std,min,5%,25%,50%,75%,95%,max
0,wind_speed,9537.0,3.890462,2.253710,0.057453,0.707309,1.978780,3.633586,5.811948,7.681033,10.297335
1,wind_direction,9537.0,248.043625,90.398140,0.254333,29.548737,230.107819,278.359100,309.678894,339.942810,359.158508
2,pressure,9537.0,1015.798279,1.441157,1012.235229,1013.328369,1014.580444,1016.014709,1017.047913,1017.749207,1018.659119
3,temperature,9537.0,18.089628,1.802176,9.848816,14.350006,17.850037,18.548798,19.013397,19.750000,21.568298
4,precipitation_rate_mm_h,9537.0,0.128979,0.181501,0.000000,0.000000,0.000000,0.000000,0.263947,0.494187,0.715116
5,hs_forecast,9537.0,1.099142,0.532715,0.040000,0.290000,0.700000,1.000000,1.520000,2.010000,2.230000
6,tp_forecast,9537.0,10.635413,3.574361,1.660000,5.130000,8.460000,10.400000,11.470000,18.629999,20.389999
7,wave_direction_forecast,9537.0,276.993652,74.890572,0.750000,196.009995,266.160004,307.519989,318.920013,335.910004,359.140015
8,swell_forecast,9537.0,0.773959,0.383903,0.027876,0.197335,0.492992,0.713023,1.067058,1.440790,1.581167


## Celda 14 — Validaciones finales

In [16]:
required_cols = [
    "run_time",
    "target_time",
    "horizon_hours",
    "zona_id",
    "lat",
    "lon",
    "source",
    "wind_speed",
    "wind_direction",
    "pressure",
    "temperature",
    "hs_forecast",
    "tp_forecast",
    "wave_direction_forecast",
    "swell_forecast",
]

for col in required_cols:
    if col not in forecast.columns:
        forecast[col] = np.nan

if forecast.empty:
    raise ValueError("forecast está vacío.")

if forecast["run_time"].isna().any() or forecast["target_time"].isna().any():
    raise ValueError("Hay run_time/target_time nulos.")

invalid = ~forecast["target_time"].between(MIN_VALID_TS, MAX_VALID_TS)

if invalid.any():
    raise ValueError(
        "Hay target_time fuera de rango: "
        f"{forecast.loc[invalid, 'target_time'].min()} - {forecast.loc[invalid, 'target_time'].max()}"
    )

if forecast["zona_id"].isna().any():
    raise ValueError("Hay zona_id nulos.")

if forecast["lat"].isna().any() or forecast["lon"].isna().any():
    raise ValueError("Hay lat/lon de zona nulos.")

# No bloquear si una parte falta parcialmente, pero sí si todo GFS o todo Wave falló.
if len(gfs_files_df) > 0 and len(gfs_summary_df) == 0:
    raise ValueError("Hay archivos GFS, pero no se procesó ninguno correctamente.")

if len(gfs_wave_files_df) > 0 and len(wave_summary_df) == 0:
    raise ValueError("Hay archivos GFS-Wave, pero no se procesó ninguno correctamente.")

# Variables clave. GFS-Wave podría tener swell con nombre no detectado, por eso no se exige swell.
if forecast["wind_speed"].isna().mean() > 0.5:
    raise ValueError("Más del 50% de wind_speed está nulo. Revisar mapeo GFS.")

if forecast["hs_forecast"].isna().mean() > 0.5:
    raise ValueError("Más del 50% de hs_forecast está nulo. Revisar mapeo GFS-Wave.")

print("Validaciones finales forecast GFS superadas.")

Validaciones finales forecast GFS superadas.


## Celda 15 — Guardar Parquet particionado

In [17]:
final_cols = [
    "run_time",
    "target_time",
    "horizon_hours",
    "zona_id",
    "nombre_zona",
    "isla",
    "municipio",
    "lat",
    "lon",
    "source",
    "forecast_type",
    "temporal_resolution",
    "wind_speed",
    "wind_direction",
    "pressure",
    "temperature",
    "precipitation_rate_mm_h",
    "u10",
    "v10",
    "hs_forecast",
    "tp_forecast",
    "wave_direction_forecast",
    "swell_forecast",
    "swell_period_forecast",
    "swell_direction_forecast",
    "gfs_grid_lat",
    "gfs_grid_lon",
    "gfs_grid_distance_km",
    "wave_grid_lat",
    "wave_grid_lon",
    "wave_grid_distance_km",
    "gfs_file",
    "gfs_wave_file",
    "year",
]

flag_cols = [c for c in forecast.columns if c.endswith("_flag")]
imputation_cols = [
    c for c in forecast.columns
    if c.endswith("_was_synthetic") or c.endswith("_imputation_method")
]
final_cols = final_cols + flag_cols + imputation_cols
final_cols = list(dict.fromkeys(final_cols))

for col in final_cols:
    if col not in forecast.columns:
        forecast[col] = np.nan

forecast_final = forecast[final_cols].copy()

remove_existing_source_partition(OUT_DIR, SOURCE_NAME)
write_partitioned_parquet(forecast_final, OUT_DIR)

print("Guardado forecast_gfs en:")
print(OUT_DIR / f"source={SOURCE_NAME}")
print("Shape:", forecast_final.shape)
display(forecast_final.head())

Eliminada partición antigua: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/forecast_gfs/source=NOAA_GFS
Guardado forecast_gfs en:
/content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/forecast_gfs/source=NOAA_GFS
Shape: (9537, 61)


,run_time,target_time,horizon_hours,zona_id,nombre_zona,isla,municipio,lat,lon,source,...,wave_direction_forecast_was_synthetic,wave_direction_forecast_imputation_method,swell_forecast_was_synthetic,swell_forecast_imputation_method,swell_period_forecast_was_synthetic,swell_period_forecast_imputation_method,swell_direction_forecast_was_synthetic,swell_direction_forecast_imputation_method,precipitation_rate_mm_h_was_synthetic,precipitation_rate_mm_h_imputation_method
0,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,NOAA_GFS,...,False,original_model_value,True,derived_from_hs_wind_direction_proxy,True,derived_from_tp_proxy,False,original_model_value,True,pressure_wind_proxy
1,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,NOAA_GFS,...,False,original_model_value,True,derived_from_hs_wind_direction_proxy,True,derived_from_tp_proxy,False,original_model_value,True,pressure_wind_proxy
2,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,CAN_EH_CHARCO_AZUL_1,Charco Azul,El Hierro,Frontera,27.7616,-18.0404,NOAA_GFS,...,False,original_model_value,True,derived_from_hs_wind_direction_proxy,True,derived_from_tp_proxy,False,original_model_value,True,pressure_wind_proxy
3,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,CAN_EH_CHARCO_DE_LOS_SARGOS,Charco de los Sargos,El Hierro,Frontera,27.7846,-18.0119,NOAA_GFS,...,False,original_model_value,True,derived_from_hs_wind_direction_proxy,True,derived_from_tp_proxy,False,original_model_value,True,pressure_wind_proxy
4,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,NOAA_GFS,...,False,original_model_value,True,derived_from_hs_wind_direction_proxy,True,derived_from_tp_proxy,False,original_model_value,True,pressure_wind_proxy


## Celda 16 — Comprobación final de lectura

In [18]:
forecast_count, forecast_sample = dataset_count_and_sample(OUT_DIR)

print("Filas guardadas forecast_gfs NOAA_GFS:", forecast_count)

if len(forecast_sample):
    display(forecast_sample)

if forecast_count == 0:
    raise ValueError("No se guardó ningún registro forecast_gfs.")

global_summary = pd.DataFrame(
    [
        {
            "table": "forecast_gfs",
            "source": SOURCE_NAME,
            "rows": forecast_count,
            "runs": forecast_final["run_time"].nunique(),
            "horizons": forecast_final["horizon_hours"].nunique(),
            "zones": forecast_final["zona_id"].nunique(),
            "target_time_min": forecast_final["target_time"].min(),
            "target_time_max": forecast_final["target_time"].max(),
            "wind_speed_missing_pct": float(forecast_final["wind_speed"].isna().mean() * 100),
            "hs_forecast_missing_pct": float(forecast_final["hs_forecast"].isna().mean() * 100),
            "tp_forecast_missing_pct": float(forecast_final["tp_forecast"].isna().mean() * 100),
            "wave_direction_forecast_missing_pct": float(forecast_final["wave_direction_forecast"].isna().mean() * 100),
        }
    ]
)

display(global_summary)

global_summary.to_csv(QC_DIR / "quality_gfs_forecast_global_summary.csv", index=False)

print("Reportes GFS:")
for p in sorted(QC_DIR.glob("quality_gfs*.csv")):
    print("-", p)

print("\nMetadatos GFS:")
for p in sorted(META_DIR.glob("gfs*.csv")):
    print("-", p)

print("\nValidación final forecast_gfs superada.")

Filas guardadas forecast_gfs NOAA_GFS: 9537


,run_time,target_time,horizon_hours,zona_id,nombre_zona,isla,municipio,lat,lon,forecast_type,...,swell_forecast_was_synthetic,swell_forecast_imputation_method,swell_period_forecast_was_synthetic,swell_period_forecast_imputation_method,swell_direction_forecast_was_synthetic,swell_direction_forecast_imputation_method,precipitation_rate_mm_h_was_synthetic,precipitation_rate_mm_h_imputation_method,source,run_date
0,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,operational,...,True,derived_from_hs_wind_direction_proxy,True,derived_from_tp_proxy,False,original_model_value,True,pressure_wind_proxy,NOAA_GFS,2026-05-08
1,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,operational,...,True,derived_from_hs_wind_direction_proxy,True,derived_from_tp_proxy,False,original_model_value,True,pressure_wind_proxy,NOAA_GFS,2026-05-08
2,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,CAN_EH_CHARCO_AZUL_1,Charco Azul,El Hierro,Frontera,27.7616,-18.0404,operational,...,True,derived_from_hs_wind_direction_proxy,True,derived_from_tp_proxy,False,original_model_value,True,pressure_wind_proxy,NOAA_GFS,2026-05-08
3,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,CAN_EH_CHARCO_DE_LOS_SARGOS,Charco de los Sargos,El Hierro,Frontera,27.7846,-18.0119,operational,...,True,derived_from_hs_wind_direction_proxy,True,derived_from_tp_proxy,False,original_model_value,True,pressure_wind_proxy,NOAA_GFS,2026-05-08
4,2026-05-08 12:00:00+00:00,2026-05-08 12:00:00+00:00,0,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,operational,...,True,derived_from_hs_wind_direction_proxy,True,derived_from_tp_proxy,False,original_model_value,True,pressure_wind_proxy,NOAA_GFS,2026-05-08


,table,source,rows,runs,horizons,zones,target_time_min,target_time_max,wind_speed_missing_pct,hs_forecast_missing_pct,tp_forecast_missing_pct,wave_direction_forecast_missing_pct
0,forecast_gfs,NOAA_GFS,9537,1,17,561,2026-05-08 12:00:00+00:00,2026-05-10 12:00:00+00:00,0.0,0.0,0.0,0.0


Reportes GFS:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_gfs_errors.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_gfs_files_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_gfs_forecast_global_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_gfs_forecast_missing_by_column.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_gfs_forecast_stats.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_gfs_forecast_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_gfs_synthetic_imputation_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_gfs_wave_errors.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/qu

## Resultado esperado

Al terminar deberían existir:

```text
silver/forecast_gfs/source=NOAA_GFS/run_date=YYYY-MM-DD/*.parquet
silver/_quality_reports/quality_gfs_forecast_summary.csv
silver/_quality_reports/quality_gfs_forecast_global_summary.csv
silver/_quality_reports/quality_gfs_files_summary.csv
silver/_quality_reports/quality_gfs_wave_files_summary.csv
silver/_metadata/gfs_forecast_files.csv
```

Comprueba especialmente:

```text
Archivos GFS procesados > 0
Archivos GFS-Wave procesados > 0
forecast shape > 0
Filas guardadas forecast_gfs NOAA_GFS > 0
Validaciones finales forecast GFS superadas
Validación final forecast_gfs superada
```

Si alguna variable aparece con muchos nulos, revisa los inventarios en `quality_gfs_errors.csv` o `quality_gfs_wave_errors.csv`; los nombres GRIB pueden variar según producto/descarga.